# Module 3 — Mini Project & Mini Assignment (LangChain)

## Mini Project — PDF Chatbot

Installs pin to the LangChain 0.3.x line, since 1.x dropped `RetrievalQA`, which the rest of this notebook relies on.

In [ ]:
!pip uninstall -y numpy scipy scikit-learn sentence-transformers
!pip install -q \
numpy==1.26.4 \
scipy==1.13.1 \
scikit-learn==1.5.1 \
sentence-transformers==3.0.1

### Build a two-page sample PDF and load it with `PyPDFLoader`

In [1]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

pdf_styles = getSampleStyleSheet()
sample_pdf = SimpleDocTemplate("sample.pdf", pagesize=letter)

page1_text = (
    "Retrieval-Augmented Generation combines a retriever and a language model so that "
    "answers are grounded in external documents instead of relying only on what the model "
    "memorized during training."
)
page2_text = (
    "FAISS is a library developed by Meta AI for fast similarity search over dense vectors. "
    "It is widely used to build the vector storage layer of a RAG pipeline because it is "
    "lightweight and runs entirely in memory."
)

sample_pdf.build([
    Paragraph(page1_text, pdf_styles["Normal"]),
    PageBreak(),
    Paragraph(page2_text, pdf_styles["Normal"]),
])

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")
raw_docs = loader.load()
raw_docs

[Document(metadata={'source': 'sample.pdf', 'page': 0}, page_content='Retrieval-Augmented Generation combines a retriever and a language model so that answers are\ngrounded in external documents instead of relying only on what the model memorized during training.\n'),
 Document(metadata={'source': 'sample.pdf', 'page': 1}, page_content='FAISS is a library developed by Meta AI for fast similarity search over dense vectors. It is widely used\nto build the vector storage layer of a RAG pipeline because it is lightweight and runs entirely in memory.\n')]


### Split the pages into smaller, overlapping chunks

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
doc_chunks = splitter.split_documents(raw_docs)
doc_chunks

[Document(metadata={'source': 'sample.pdf', 'page': 0}, page_content='Retrieval-Augmented Generation combines a retriever and a language model so that answers are'),
 Document(metadata={'source': 'sample.pdf', 'page': 0}, page_content='grounded in external documents instead of relying only on what the model memorized during training.'),
 Document(metadata={'source': 'sample.pdf', 'page': 1}, page_content='FAISS is a library developed by Meta AI for fast similarity search over dense vectors. It is widely used'),
 Document(metadata={'source': 'sample.pdf', 'page': 1}, page_content='to build the vector storage layer of a RAG pipeline because it is lightweight and runs entirely in memory.')]


### Embed the chunks and store them in a FAISS vector store

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(doc_chunks, embedder)

### Build a retriever and load a Hugging Face model to answer questions

In [1]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

gen_pipeline = pipeline("text-generation", model="google/flan-t5-base", max_new_tokens=100)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

Device set to use cuda:0
Note: T5ForConditionalGeneration is an encoder-decoder model, so it isn't in the standard causal-LM text-generation model list — this still works via the pipeline wrapper.


### Wire the retriever and the LLM together into a `RetrievalQA` chain

In [ ]:
from langchain.chains import RetrievalQA

pdf_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
)

### Ask a few questions and inspect the retrieved context alongside each answer

In [1]:
sample_questions = [
    "What does RAG combine to ground its answers?",
    "Who developed FAISS?",
    "Why is FAISS considered lightweight?",
]

for q in sample_questions:
    result = pdf_qa_chain.invoke({"query": q})
    context_used = " ".join(doc.page_content for doc in result["source_documents"])

    print("Question:", q)
    print("Retrieved Context:", context_used)
    print("Generated Answer:", result["result"])
    print("-" * 80)

Question: What does RAG combine to ground its answers?
Retrieved Context: to build the vector storage layer of a RAG pipeline because it is lightweight and runs entirely in memory. Retrieval-Augmented Generation combines a retriever and a language model so that answers are
Generated Answer: Retrieval-Augmented Generation combines a retriever and a language model so that answers are grounded in external documents.
--------------------------------------------------------------------------------
Question: Who developed FAISS?
Retrieved Context: FAISS is a library developed by Meta AI for fast similarity search over dense vectors. It is widely used to build the vector storage layer of a RAG pipeline because it is lightweight and runs entirely in memory.
Generated Answer: FAISS is a library developed by Meta AI.
--------------------------------------------------------------------------------
Question: Why is FAISS considered lightweight?
Retrieved Context: FAISS is a library developed by Me

## Mini Assignment — Multi-format Chatbot (PDF, TXT, CSV)

### Install dependencies

In [ ]:
!pip install -q "langchain==0.3.7" "langchain-community==0.3.7" "langchain-huggingface==0.1.2" \
"langchain-text-splitters==0.3.2" pypdf faiss-cpu reportlab sentence-transformers transformers pandas

### Build a sample PDF, TXT, and CSV source file

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

policy_styles = getSampleStyleSheet()
policy_pdf = SimpleDocTemplate("policy.pdf", pagesize=letter)

leave_policy_p1 = (
    "Employees are entitled to twenty four days of paid leave every calendar year, which "
    "resets on the first of January."
)
leave_policy_p2 = (
    "Any unused leave beyond five days at year end is forfeited and cannot be carried "
    "forward to the next year."
)

policy_pdf.build([
    Paragraph(leave_policy_p1, policy_styles["Normal"]),
    PageBreak(),
    Paragraph(leave_policy_p2, policy_styles["Normal"]),
])

reimbursement_note = (
    "The reimbursement process requires employees to submit original bills within thirty "
    "days of the expense along with a filled reimbursement form."
)
with open("guidelines.txt", "w") as f:
    f.write(reimbursement_note)

import pandas as pd

faq_table = pd.DataFrame({
    "question": [
        "How do I apply for leave?",
        "What is the notice period for resignation?",
    ],
    "answer": [
        "Leave can be applied through the HR portal at least two days in advance.",
        "The standard notice period for resignation is thirty days.",
    ],
})
faq_table.to_csv("faq.csv", index=False)

### Load each file type with its matching LangChain loader

In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader

pdf_docs = PyPDFLoader("policy.pdf").load()
txt_docs = TextLoader("guidelines.txt").load()
csv_docs = CSVLoader("faq.csv").load()

combined_docs = pdf_docs + txt_docs + csv_docs
combined_docs

[Document(metadata={'source': 'policy.pdf', 'page': 0}, page_content='Employees are entitled to twenty four days of paid leave every calendar year, which resets on the first\nof January.\n'),
 Document(metadata={'source': 'policy.pdf', 'page': 1}, page_content='Any unused leave beyond five days at year end is forfeited and cannot be carried forward to the next\nyear.\n'),
 Document(metadata={'source': 'guidelines.txt'}, page_content='The reimbursement process requires employees to submit original bills within thirty days of the expense along with a filled reimbursement form.'),
 Document(metadata={'source': 'faq.csv', 'row': 0}, page_content='question: How do I apply for leave?\nanswer: Leave can be applied through the HR portal at least two days in advance.'),
 Document(metadata={'source': 'faq.csv', 'row': 1}, page_content='question: What is the notice period for resignation?\nanswer: The standard notice period for resignation is thirty days.')]


### Chunk everything, keeping each chunk's source/page/row metadata intact

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
combined_chunks = splitter.split_documents(combined_docs)
combined_chunks

[Document(metadata={'source': 'policy.pdf', 'page': 0}, page_content='Employees are entitled to twenty four days of paid leave every calendar year, which resets on the first\nof January.'),
 Document(metadata={'source': 'policy.pdf', 'page': 1}, page_content='Any unused leave beyond five days at year end is forfeited and cannot be carried forward to the next\nyear.'),
 Document(metadata={'source': 'guidelines.txt'}, page_content='The reimbursement process requires employees to submit original bills within thirty days of the expense along with a'),
 Document(metadata={'source': 'guidelines.txt'}, page_content='along with a filled reimbursement form.'),
 Document(metadata={'source': 'faq.csv', 'row': 0}, page_content='question: How do I apply for leave?\nanswer: Leave can be applied through the HR portal at least two days in advance.'),
 Document(metadata={'source': 'faq.csv', 'row': 1}, page_content='question: What is the notice period for resignation?\nanswer: The standard notice perio

### Embed all chunks and build a single FAISS store across the three sources

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
combined_vectorstore = FAISS.from_documents(combined_chunks, embedder)

### Build the retriever + LLM, and assemble the `RetrievalQA` chain

In [1]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA

combined_retriever = combined_vectorstore.as_retriever(search_kwargs={"k": 3})

gen_pipeline = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=100)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

combined_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=combined_retriever,
    return_source_documents=True,
)

Device set to use cuda:0


### Ask a question and show which source/page/row each retrieved chunk came from (bonus)

In [1]:
user_question = input("Enter your question: ")

result = combined_qa_chain.invoke({"query": user_question})

print("Question:", user_question)
print("Generated Answer:", result["result"])
print()
print("Retrieved Chunks:")
for doc in result["source_documents"]:
    src = doc.metadata.get("source")
    loc = doc.metadata.get("page", doc.metadata.get("row"))
    print(f"- Source: {src}, Page/Row: {loc}")
    print(f"  Text: {doc.page_content}")

Enter your question: How do I apply for leave?
Question: How do I apply for leave?
Generated Answer: Leave can be applied through the HR portal at least two days in advance.

Retrieved Chunks:
- Source: faq.csv, Page/Row: 0
  Text: question: How do I apply for leave?
answer: Leave can be applied through the HR portal at least two days in advance.
- Source: policy.pdf, Page/Row: 1
  Text: Any unused leave beyond five days at year end is forfeited and cannot be carried forward to the next
year.
- Source: policy.pdf, Page/Row: 0
  Text: Employees are entitled to twenty four days of paid leave every calendar year, which resets on the first
of January.
